# 01 — Download BRFSS 2023 and verify the codings

This notebook caches the BRFSS 2023 public-use file and the official codebook, then
**verifies every coding the analysis depends on against the codebook itself** before
any recode. Reproducibility hinges on this step; we never trust a remembered code.

Files cached under `../data/raw/brfss/2023/`:

- `LLCP2023XPT.zip` → `LLCP2023.XPT` — the annual landline+cellphone (LLCP) public-use file (CDC).
- `codebook23_llcp-v2-508.zip` → `USCODE23_LLCP_021924.HTML` — the variable codebook.
- `calc_vars_2023.pdf` — the Calculated Variables report.

We pull only the columns the analysis uses and cache a slim parquet
(`../data/derived/brfss_2023_raw_subset.parquet`) so later notebooks load in seconds.

In [1]:
import os, zipfile, requests
import numpy as np
import pandas as pd
import pyreadstat

import recode as rc

DATA = os.path.abspath(os.path.join('..', 'data'))
RAW = os.path.join(DATA, 'raw', 'brfss', '2023')
DERIVED = os.path.join(DATA, 'derived')
os.makedirs(RAW, exist_ok=True)
os.makedirs(DERIVED, exist_ok=True)

FILES = {
    'LLCP2023XPT.zip':
        'https://www.cdc.gov/brfss/annual_data/2023/files/LLCP2023XPT.zip',
    'codebook23_llcp-v2-508.zip':
        'https://www.cdc.gov/brfss/annual_data/2023/zip/codebook23_llcp-v2-508.zip',
    'calc_vars_2023.pdf':
        'https://www.cdc.gov/brfss/annual_data/2023/pdf/2023-calculated-variables-version4-508.pdf',
}

def download(name, url):
    dest = os.path.join(RAW, name)
    if os.path.exists(dest):
        print(f'cached  {name} ({os.path.getsize(dest)/1e6:.1f} MB)')
        return dest
    print(f'fetching {name} ...')
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        f.write(r.content)
    print(f'saved   {name} ({os.path.getsize(dest)/1e6:.1f} MB)')
    return dest

for name, url in FILES.items():
    download(name, url)

cached  LLCP2023XPT.zip (93.2 MB)
cached  codebook23_llcp-v2-508.zip (0.1 MB)
cached  calc_vars_2023.pdf (1.2 MB)


## Extract and load the needed columns

The XPT is ~700 MB uncompressed; we read only the columns the analysis uses.

In [2]:
xpt = os.path.join(RAW, 'LLCP2023.XPT')
if not os.path.exists(xpt):
    z = zipfile.ZipFile(os.path.join(RAW, 'LLCP2023XPT.zip'))
    member = z.namelist()[0]               # 'LLCP2023.XPT ' (note trailing space)
    with z.open(member) as src, open(xpt, 'wb') as out:
        out.write(src.read())

df, meta = pyreadstat.read_xport(xpt, usecols=rc.COLUMNS)
print('rows:', len(df), ' columns:', list(df.columns))
df.head(3)

rows: 433323  columns: ['_STATE', '_PSU', 'SEXVAR', 'GENHLTH', 'MENTHLTH', 'MARITAL', 'EMPLOY1', 'INCOME3', 'BLIND', '_STSTR', '_IMPRACE', '_LLCPWT', '_AGEG5YR', '_AGE80', '_EDUCAG']


,_STATE,_PSU,SEXVAR,GENHLTH,MENTHLTH,MARITAL,EMPLOY1,INCOME3,BLIND,_STSTR,_IMPRACE,_LLCPWT,_AGEG5YR,_AGE80,_EDUCAG
0,1.0,2.023000e+09,2.0,2.0,88.0,1.0,7.0,99.0,2.0,11011.0,1.0,605.427887,13.0,80.0,3.0
1,1.0,2.023000e+09,2.0,2.0,88.0,2.0,7.0,99.0,2.0,11012.0,1.0,1121.992705,13.0,80.0,3.0
2,1.0,2.023000e+09,2.0,4.0,2.0,3.0,7.0,2.0,1.0,11011.0,2.0,600.963308,13.0,80.0,2.0


## Verify the income coding against the codebook

The whole project rests on `INCOME3` storing **Refused (99)** and **Don't know /
Not sure (77)** as *distinct* codes, with brackets 1–11. We confirm the frequencies
reproduce the codebook's published counts.

In [3]:
inc = pd.to_numeric(df['INCOME3'], errors='coerce')
tab = inc.value_counts(dropna=False).sort_index()
tab = tab.rename_axis('INCOME3').reset_index(name='frequency')
tab['pct'] = 100 * tab['frequency'] / len(df)
print(tab.to_string(index=False))
print()
print(f"Refused (99):     {int((inc==99).sum()):>7,}  ({100*(inc==99).mean():.2f}%)")
print(f"Don't know (77):  {int((inc==77).sum()):>7,}  ({100*(inc==77).mean():.2f}%)")
print('Codebook check (expected): 99 -> 42,232 (9.93%);  77 -> 36,316 (8.54%)')

 INCOME3  frequency       pct
     1.0       9280  2.141590
     2.0       9907  2.286285
     3.0      12867  2.969379
     4.0      18202  4.200562
     5.0      38508  8.886673
     6.0      47502 10.962261
     7.0      57896 13.360934
     8.0      49131 11.338193
     9.0      52284 12.065826
    10.0      24353  5.620057
    11.0      26770  6.177840
    77.0      36316  8.380815
    99.0      42232  9.746079
     NaN       8075  1.863506

Refused (99):      42,232  (9.75%)
Don't know (77):   36,316  (8.38%)
Codebook check (expected): 99 -> 42,232 (9.93%);  77 -> 36,316 (8.54%)


The bracket dollar boundaries we will use for the grouped-data likelihood (verified
against the codebook, code → interval):

In [4]:
for code, (lo, hi) in rc.INCOME_BRACKETS_USD.items():
    hi_s = '+inf (open top bracket)' if not np.isfinite(hi) else f'{hi:,.0f}'
    print(f'  {code:>2}: (${lo:,.0f}, ${hi_s})')

   1: ($0, $10,000)
   2: ($10,000, $15,000)
   3: ($15,000, $20,000)
   4: ($20,000, $25,000)
   5: ($25,000, $35,000)
   6: ($35,000, $50,000)
   7: ($50,000, $75,000)
   8: ($75,000, $100,000)
   9: ($100,000, $150,000)
  10: ($150,000, $200,000)
  11: ($200,000, $+inf (open top bracket))


## Verify the outcome coding (`MENTHLTH`)

`MENTHLTH` is days of poor mental health in the past 30: 1–30 = days, 88 = none,
77 = Don't know, 99 = Refused. Frequent mental distress (FMD) = ≥ 14 days.

In [5]:
mh = pd.to_numeric(df['MENTHLTH'], errors='coerce')
print('1-30 days :', int(((mh>=1)&(mh<=30)).sum()))
print('88 (none) :', int((mh==88).sum()))
print('77 (DK)   :', int((mh==77).sum()))
print('99 (Ref)  :', int((mh==99).sum()))
days, valid = rc.recode_menthlth(df['MENTHLTH'])
fmd = rc.fmd_from_days(days)
print(f'\nFMD (>=14 days), unweighted prevalence among valid: '
      f'{np.nanmean(fmd[valid]):.4f}')

1-30 days : 168189
88 (none) : 257026
77 (DK)   : 5992
99 (Ref)  : 2113

FMD (>=14 days), unweighted prevalence among valid: 0.1366


## Verify the second outcome (`BLIND`, difficulty seeing)

Disability core item: *"Are you blind or do you have serious difficulty seeing, even
when wearing glasses?"* — 1 = Yes, 2 = No, 7 = Don't know, 9 = Refused. Used in
notebook 05.

In [6]:
bl = pd.to_numeric(df['BLIND'], errors='coerce')
print('1 Yes     :', int((bl==1).sum()))
print('2 No      :', int((bl==2).sum()))
print('7 DK      :', int((bl==7).sum()))
print('9 Refused :', int((bl==9).sum()))
print('Codebook check (expected): Yes -> 22,190; No -> 395,423; DK -> 1,113; Ref -> 517')

1 Yes     : 22190
2 No      : 395423
7 DK      : 1113
9 Refused : 517
Codebook check (expected): Yes -> 22,190; No -> 395,423; DK -> 1,113; Ref -> 517


## Cache a slim subset for the downstream notebooks

In [7]:
out = os.path.join(DERIVED, 'brfss_2023_raw_subset.parquet')
df.to_parquet(out, index=False)
print('wrote', out, f'({os.path.getsize(out)/1e6:.1f} MB)')

wrote /mnt/share/homes/abie/projects/2026/ai_assisted_us_health_data_analysis/data/derived/brfss_2023_raw_subset.parquet (6.3 MB)
